In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split 
import joblib

In [2]:
final_pipeline = joblib.load('../models/loan_default_pipeline.joblib')
feature_names = joblib.load('../models/feature_names.joblib')
print(final_pipeline)

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('classifier',
                 LogisticRegression(class_weight='balanced', max_iter=1000,
                                    random_state=42))])


In [3]:
df = pd.read_csv("../data/loans_cleaned.csv")
X = df.drop(columns=['not.fully.paid'])
y = df['not.fully.paid']

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [5]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (7662, 16)
X_test shape: (1916, 16)


In [6]:
preprocessed_X_train = final_pipeline[:-1].transform(X_train)
preprocessed_X_test = final_pipeline[:-1].transform(X_test)
classifier = final_pipeline.named_steps['classifier']

print("preprocessed_X_train shape:", preprocessed_X_train.shape)
print("classifier:", classifier)


preprocessed_X_train shape: (7662, 16)
classifier: LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)


In [7]:
import shap

explainer = shap.LinearExplainer(classifier, preprocessed_X_test)
print("Explainer created:", explainer)

shap_values = explainer(preprocessed_X_test)
print("SHAP values computed. SHape:", shap_values.values.shape)

c:\Users\anjum\Desktop\Loan\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Background dataset has 1916 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=1916 when initializing the masker.


Explainer created: <shap.explainers._linear.LinearExplainer object at 0x000001E2FD1F4620>
SHAP values computed. SHape: (1916, 16)


In [8]:
final_pred_proba = final_pipeline.predict_proba(X_test)[:, 1]

i = 0
print(f"\nExplanation for test applicant #{i}:")
for name, val, shap_val in zip(feature_names, X_test.iloc[i], shap_values.values[i]):
    print(f"  {name}: value={val:.3f}, shap_contribution={shap_val:.4f}")

print("\nBase value:", shap_values.base_values[i])
print("Predicted probability:", final_pred_proba[i])


Explanation for test applicant #0:
  log.annual.inc: value=11.035, shap_contribution=-0.0033
  dti: value=15.380, shap_contribution=-0.0110
  fico: value=647.000, shap_contribution=0.7144
  revol.bal: value=11214.000, shap_contribution=-0.0111
  revol.util: value=70.100, shap_contribution=0.0586
  inq.last.6mths: value=0.000, shap_contribution=-0.2231
  delinq.2yrs: value=0.000, shap_contribution=0.0116
  pub.rec: value=0.000, shap_contribution=-0.0148
  purpose_credit_card: value=0.000, shap_contribution=0.0822
  purpose_debt_consolidation: value=1.000, shap_contribution=-0.1231
  purpose_educational: value=0.000, shap_contribution=-0.0009
  purpose_home_improvement: value=0.000, shap_contribution=-0.0128
  purpose_major_purchase: value=0.000, shap_contribution=0.0046
  purpose_small_business: value=0.000, shap_contribution=-0.0299
  revol.bal_to_income: value=0.181, shap_contribution=-0.0076
  years_with_cr_line: value=5.096, shap_contribution=-0.0548

Base value: -0.207941251056244